In [13]:
import gc
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix  # Toegevoegd voor metrics
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# ==========================================
# CONFIGURATIE
# ==========================================
DATA_PATH = "transformed_data_sampled.csv"
TARGET_COL = "status"  # De doelvariabele die nu 0, 1, 2... bevat
IS_CLASSIFICATION = True  # Moet True zijn voor statuscode-voorspelling

# ==========================================
# 1. GEHEUGENEFFICIËNT DATA INLADEN (DOWNCASTING)
# ==========================================
print("1. Dtypes analyseren voor geheugenbesparing...")
sample = pd.read_csv(DATA_PATH, nrows=100)

dtypes = {}
for col in sample.columns:
    if sample[col].dtype == "float64":
        dtypes[col] = "float32"
    elif sample[col].dtype == "int64":
        dtypes[col] = "int32"
    else:
        dtypes[col] = sample[col].dtype

print("-> Data inladen met geoptimaliseerde dtypes...")
df = pd.read_csv(DATA_PATH, dtype=dtypes)

# Splits direct in X en y
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

del df
gc.collect()

# ==========================================
# 2. TRAIN / VALIDATION SPLIT
# ==========================================
print("2. Dataset opsplitsen...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# === DYNAMISCH AANTAL KLASSEN BEPALEN ===
num_classes = int(y_train.max() + 1)
print(f"-> Aantal gedetecteerde klassen (statuscodes) voor training: {num_classes}")

# We bewaren y_val even voor de metrics straks, maar we gooien X en y weg voor het geheugen
del X, y
gc.collect()

# ==========================================
# 3. CLASS IMBALANCE OPLOSSEN MET SAMPLE WEIGHTS
# ==========================================
print("3. Class weights berekenen (geen SMOTE / ROS)...")

class_counts = y_train.value_counts().sort_index()

# inverse frequency (stabiele variant)
class_weights = {
    cls: len(y_train) / (len(class_counts) * count)
    for cls, count in class_counts.items()
}

print("-> Class distribution:")
print(class_counts)

print("\n-> Class weights:")
for k, v in class_weights.items():
    print(f"Class {k}: {v:.6f}")

# sample weights per training row
sample_weights = y_train.map(class_weights).values

print("\n-> Native XGBoost DMatrix aanmaken met sample weights...")
dtrain = xgb.DMatrix(
    X_train,
    label=y_train,
    weight=sample_weights
)

dval = xgb.DMatrix(X_val, label=y_val)

print("-> DMatrix klaar.")
# ==========================================
# 4. PARAMETERS OPTIMALISEREN VOOR MULTICLASS & 8GB RAM
# ==========================================
objective = "multi:softprob" if IS_CLASSIFICATION else "reg:squarederror"
eval_metric = "mlogloss" if IS_CLASSIFICATION else "rmse"

params = {
    "objective": objective,
    "num_class": num_classes,
    "tree_method": "hist",
    "max_depth": 6,             # Verlaagd van 6 naar 4 tegen overfitting
    "min_child_weight": 5,     # Toegevoegd: dwingt grotere groepen af per splitsing
    "learning_rate": 0.05,
    "eval_metric": eval_metric,
    "subsample": 0.7,           # Iets scherper gezet (70% ipv 80%)
    "colsample_bytree": 0.7,    # Iets scherper gezet (70% ipv 80%)
    "verbosity": 1,
    "max_delta_step": 1
}

# ==========================================
# 5. MODEL TRAINEN
# ==========================================
print("4. Starten met trainen...")
evallist = [(dval, "validation"), (dtrain, "train")]
num_round = 2000  

bst = xgb.train(
    params,
    dtrain,
    num_boost_round=num_round,
    evals=evallist,
    early_stopping_rounds=50,  
    verbose_eval=50,  
)

print("\nTraining succesvol afgerond!")

# ==========================================
# 6. METRICS BEREKENEN EN TONEN
# ==========================================
print("\n=== MODEL PERFORMANCE METRICS ===")

# 1. Beste iteratie ophalen
best_iteration = bst.best_iteration
best_score = bst.best_score
print(f"Beste iteratie (boom): {best_iteration} (Validation mlogloss: {best_score:.4f})")

# 2. Voorspellingen doen op de validatiedata (geeft kansen per klasse)
# iteration_range zorgt ervoor dat we de bomen tot en met de beste iteratie gebruiken
preds_prob = bst.predict(dval, iteration_range=(0, best_iteration + 1))

# Omzetten van kansen naar de klasse met de hoogste kans
y_pred = np.argmax(preds_prob, axis=1)

# 3. Classificatierapport genereren (voor F1-score macro en Recall macro)
report = classification_report(y_val, y_pred, output_dict=True)

f1_macro = report["macro avg"]["f1-score"]
recall_macro = report["macro avg"]["recall"]

print(f"F1-Score (Macro Average): {f1_macro:.4f}")
print(f"Recall (Macro Average):   {recall_macro:.4f}")

# Voor een mooi overzicht printen we ook het complete rapport
print("\nVolledig Classificatierapport:")
print(classification_report(y_val, y_pred))

# 4. Confusion Matrix berekenen en tonen
print("Confusion Matrix:")
conf_matrix = confusion_matrix(y_val, y_pred)
print(conf_matrix)

# Optioneel: Model opslaan
# bst.save_model("xgboost_multiclass_statuscodes.json")

1. Dtypes analyseren voor geheugenbesparing...
-> Data inladen met geoptimaliseerde dtypes...
2. Dataset opsplitsen...
-> Aantal gedetecteerde klassen (statuscodes) voor training: 7
3. Class weights berekenen (geen SMOTE / ROS)...
-> Class distribution:
status
0    36885
1       93
2        1
3    10680
4    70384
5     1941
6       16
Name: count, dtype: int64

-> Class weights:
Class 0: 0.464765
Class 1: 184.331797
Class 2: 17142.857143
Class 3: 1.605136
Class 4: 0.243562
Class 5: 8.831972
Class 6: 1071.428571

-> Native XGBoost DMatrix aanmaken met sample weights...
-> DMatrix klaar.
4. Starten met trainen...
[0]	validation-mlogloss:1.91635	train-mlogloss:1.91278
[50]	validation-mlogloss:1.27345	train-mlogloss:0.90845
[100]	validation-mlogloss:1.13000	train-mlogloss:0.71276
[150]	validation-mlogloss:1.07787	train-mlogloss:0.65573
[200]	validation-mlogloss:1.05105	train-mlogloss:0.63056
[250]	validation-mlogloss:1.03546	train-mlogloss:0.61553
[300]	validation-mlogloss:1.02453	train-m

/home/zacky/opt/Stage-opdracht-2de-jaar/fuzzing-project/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/zacky/opt/Stage-opdracht-2de-jaar/fuzzing-project/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/zacky/opt/Stage-opdracht-2de-jaar/fuzzing-project/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control thi

In [10]:
# Train een mini-model met 5 bomen om de belangrijkste feature te vinden
mini_bst = xgb.train(params, dtrain, num_boost_round=5)
importance = mini_bst.get_score(importance_type="gain")
print("Belangrijkste features:", sorted(importance.items(), key=lambda x: x[1], reverse=True)[:3])


Belangrijkste features: [('type_GET', 7014.96728515625), ('body_MISSING', 222.2132568359375), ('type_POST', 112.40474700927734)]
